# 02 — Verificação dos resultados do artigo

Este caderno confere, contra os dados publicados neste repositório, **cada número que o
artigo afirma** e regenera a **Figura 1**, que é a única figura do texto submetido.

O artigo relata as forças sazonais no corpo do texto, não em tabela. A verificação, portanto,
tem como alvo as afirmações do manuscrito — e não um artefato intermediário:

| Afirmação no artigo | Etapa |
| --- | --- |
| 658 de 736 combinações classe–data retiveram um extremo | 3 |
| 160, 173, 165 e 160 datas para as classes 02, 09, 10 e 11 | 3 |
| as 16 medianas de $F_S$ por classe e domínio | 4 |
| distâncias medianas entre extremos de 222 a 774 km | 5 |
| Figura 1 — componentes STL das quatro classes | 6 |

Todas as etapas terminam em `assert`. Se qualquer valor divergir do que está publicado, o
caderno falha em vez de produzir silenciosamente um resultado diferente.

## Ambiente

```bash
conda env create -f environment.yml && conda activate worcap-endmembers
```

Os `.parquet` foram escritos com `pyarrow==22.0.0`, versão fixada em `requirements.txt` e
`environment.yml`; versões anteriores falham com `Repetition level histogram size mismatch`.
`.parquet`, para que o caderno permaneça executável fora do ambiente fixado.

## 1. Localização, ambiente e integridade dos dados

In [ ]:
from __future__ import annotations

import hashlib
import importlib.util
import sys
from pathlib import Path

import matplotlib
import numpy as np
import pandas as pd
import pyarrow


def repository_root(start: str | Path | None = None) -> Path:
    '''Localiza a raiz do repositório subindo a partir do diretório de trabalho.

    Todos os caminhos deste caderno derivam daqui; nada é absoluto, de modo que ele
    funciona em qualquer máquina, desde que executado de dentro do repositório clonado.
    '''
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "config" / "study.yaml").is_file() and (candidate / "data" / "analysis").is_dir():
            return candidate
    raise FileNotFoundError(
        f"Raiz do repositório não encontrada a partir de {current}.\n"
        "Execute este caderno de dentro do repositório clonado — ele depende de\n"
        "config/, data/ e scripts/, que não acompanham o .ipynb isolado."
    )


ROOT = repository_root()
ANALYSIS = ROOT / "data" / "analysis" / "spectrotemporal"
OUTPUT = ROOT / "outputs" / "verificacao_artigo"
OUTPUT.mkdir(parents=True, exist_ok=True)


def rel(path: Path) -> str:
    '''Caminhos relativos à raiz, para que a saída não exponha a máquina de origem.'''
    try:
        return str(Path(path).resolve().relative_to(ROOT)).replace("\\", "/")
    except ValueError:
        return str(path)



print("saída deste caderno :", rel(OUTPUT))
print()
print("python    ", sys.version.split()[0])
print("numpy     ", np.__version__)
print("pandas    ", pd.__version__)
print("matplotlib", matplotlib.__version__)


In [ ]:
def sha256_of(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


manifest = pd.read_csv(ANALYSIS / "integrity_sha256.csv")
rows = []
for record in manifest.itertuples(index=False):
    target = ANALYSIS / record.file
    if not target.is_file():
        rows.append({"arquivo": record.file, "situação": "AUSENTE"})
        continue
    matches = sha256_of(target) == record.sha256 and target.stat().st_size == int(record.bytes)
    rows.append({"arquivo": record.file, "situação": "confere" if matches else "DIVERGENTE"})

integrity = pd.DataFrame(rows)
print("arquivos verificados:", len(integrity), "->", integrity["situação"].value_counts().to_dict())

divergent = integrity[integrity["situação"] != "confere"]
assert divergent.empty, f"integridade quebrada:\n{divergent}"
print("todos os arquivos de entrada conferem com o manifesto publicado")

## 2. Carregamento

`spectrotemporal_components` guarda, para cada classe, data e banda, a decomposição STL em
baixa frequência, componente sazonal e resíduo. As demais tabelas trazem as métricas
agregadas, que as etapas seguintes reconferem a partir dos componentes brutos.

In [ ]:

def read_table(path: Path) -> pd.DataFrame:
    frame = pd.read_parquet(path)
    if "class_code" in frame.columns:
        frame["class_code"] = frame["class_code"].astype(str).str.zfill(2)
    return frame
components = read_table(ANALYSIS / "spectrotemporal_components.parquet")
domain_metrics = read_table(ANALYSIS / "spectrotemporal_domain_metrics.parquet")
turnover = read_table(ANALYSIS / "representative_turnover.parquet")

CLASSES = ("02", "09", "10", "11")
CLASS_NAMES = {
    "02": "Vegetação Secundária",
    "09": "Silvicultura",
    "10": "Pastagem Arbustiva/Arbórea",
    "11": "Pastagem Herbácea",
}

print("componentes      :", components.shape)
print("métricas/domínio :", domain_metrics.shape)
print("rotatividade     :", turnover.shape)


## 3. Completude: 658 de 736 combinações

> *"Das 736 combinações classe–data, 658 retiveram um extremo compatível com vegetação ativa.
> As classes Vegetação Secundária, Silvicultura, Pastagem Arbustiva/Arbórea e Pastagem
> Herbácea contribuíram, respectivamente, com 160, 173, 165 e 160 datas."*

In [ ]:
DATAS_NO_ARTIGO = {"02": 160, "09": 173, "10": 165, "11": 160}
TOTAL_DATAS = 184

primary = domain_metrics[domain_metrics["variant"].eq("primary")]
observadas = {
    code: int(primary.loc[primary["class_code"].eq(code), "observed_dates"].iloc[0])
    for code in CLASSES
}

completude = pd.DataFrame(
    {
        "classe": [f"{c} — {CLASS_NAMES[c]}" for c in CLASSES],
        "datas (dados)": [observadas[c] for c in CLASSES],
        "datas (artigo)": [DATAS_NO_ARTIGO[c] for c in CLASSES],
    }
)
print(completude.to_string(index=False))
print()

assert observadas == DATAS_NO_ARTIGO, f"datas por classe divergem: {observadas}"

combinacoes = len(CLASSES) * TOTAL_DATAS
retidas = sum(observadas.values())
print(f"combinações classe–data : {combinacoes}  (artigo: 736)")
print(f"retidas com extremo     : {retidas}  (artigo: 658)")
assert combinacoes == 736 and retidas == 658
print()
print("completude confere com o artigo")

## 4. As 16 medianas de $F_S$

> *"As medianas de FS foram, para VIS, red edge, NIR e SWIR, respectivamente: 0,046, 0,435,
> 0,494 e 0,143, Vegetação Secundária; 0,175, 0,331, 0,350 e 0,318, Silvicultura; 0,242,
> 0,481, 0,439 e 0,343, Pastagem Arbustiva/Arbórea; e 0,295, 0,551, 0,583 e 0,010, Pastagem
> Herbácea."*

Os valores são **rederivados dos componentes STL brutos**, sem usar as métricas já agregadas,
para que a métrica apareça sendo calculada e não apenas relida. A força sazonal segue
Wang et al. (2006) e usa somente as datas efetivamente observadas — as interpoladas entram no
STL, mas não nas métricas:

$$F_S = \max\left(0,\ 1 - \frac{\operatorname{Var}(R)}{\operatorname{Var}(S + R)}\right)$$

Cada domínio é resumido pela mediana de suas bandas.

In [ ]:
DOMAINS = {
    "VIS": ["B02", "B03", "B04"],
    "RED_EDGE": ["B05", "B06", "B07"],
    "NIR": ["B08", "B8A"],
    "SWIR": ["B11", "B12"],
}
FS_NO_ARTIGO = {
    "02": [0.046, 0.435, 0.494, 0.143],
    "09": [0.175, 0.331, 0.350, 0.318],
    "10": [0.242, 0.481, 0.439, 0.343],
    "11": [0.295, 0.551, 0.583, 0.010],
}


def seasonal_strength(component: np.ndarray, residual: np.ndarray) -> float:
    denominator = float(np.var(component + residual, ddof=1))
    if denominator <= 1e-15:
        return np.nan
    return float(max(0.0, 1.0 - float(np.var(residual, ddof=1)) / denominator))


observed = components[~components["imputed_for_stl"].astype(bool)]
print(f"linhas observadas usadas nas métricas: {len(observed)} de {len(components)}")

per_band = []
for (class_code, band), group in observed.groupby(["class_code", "band"]):
    per_band.append(
        {
            "class_code": class_code,
            "band": band,
            "F_S": seasonal_strength(
                group["seasonal"].to_numpy(dtype=float), group["residual"].to_numpy(dtype=float)
            ),
        }
    )
per_band = pd.DataFrame(per_band)

linhas = []
for code in CLASSES:
    for posicao, (domain, bands) in enumerate(DOMAINS.items()):
        subset = per_band[per_band["class_code"].eq(code) & per_band["band"].isin(bands)]
        rederivado = float(subset["F_S"].median())
        linhas.append(
            {
                "classe": code,
                "domínio": domain,
                "F_S rederivado": rederivado,
                "F_S no artigo": FS_NO_ARTIGO[code][posicao],
                "|Δ| após arredondar": abs(round(rederivado, 3) - FS_NO_ARTIGO[code][posicao]),
            }
        )
comparacao = pd.DataFrame(linhas)
print()
print(comparacao.to_string(index=False))
print()

divergentes = comparacao[comparacao["|Δ| após arredondar"] > 1e-9]
assert divergentes.empty, f"F_S divergente do artigo:\n{divergentes}"
print("as 16 medianas de F_S publicadas foram reproduzidas a partir dos componentes brutos")

## 5. Rotatividade espacial dos extremos

> *"A posição dos extremos não foi mantida entre datas (distâncias medianas de 222 a 774 km),
> de modo que as trajetórias concatenam localizações distintas."*

Esta é a limitação central declarada no artigo: o "representante" de cada data pode ser um
pixel diferente, e a trajetória não é a série de um pixel fixo.

In [ ]:
tv = turnover[turnover["class_code"].isin(CLASSES)].copy()
tv = tv.sort_values("class_code")
print(
    tv[
        [
            "class_code",
            "valid_dates",
            "unique_representative_pixels",
            "median_adjacent_distance_km",
        ]
    ].to_string(index=False)
)

menor = float(tv["median_adjacent_distance_km"].min())
maior = float(tv["median_adjacent_distance_km"].max())
print()
print(f"faixa das distâncias medianas: {menor:.0f} a {maior:.0f} km  (artigo: 222 a 774 km)")

assert round(menor) == 222 and round(maior) == 774, "a faixa de distâncias diverge do artigo"
assert int(tv["valid_dates"].sum()) == 658
print("a rotatividade espacial confere com o artigo")

## 6. Figura 1

A figura é construída pela mesma função que gerou o artigo
(`scripts/build_spectrotemporal_outputs.py`), carregada diretamente do repositório para que
não exista uma segunda implementação capaz de divergir da primeira.

In [ ]:
spec = importlib.util.spec_from_file_location(
    "build_spectrotemporal_outputs", ROOT / "scripts" / "build_spectrotemporal_outputs.py"
)
builder = importlib.util.module_from_spec(spec)
spec.loader.exec_module(builder)

builder.configure_style()
builder.build_figure(components, OUTPUT)
reportadas = builder.build_reported_metrics(domain_metrics, OUTPUT)

for item in sorted(OUTPUT.iterdir()):
    print(f"{item.stat().st_size:>8} bytes  {rel(item)}")
print()
print(reportadas.to_string(index=False))

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(OUTPUT / "figura_01_decomposicao_espectrotemporal.png"), width=900))

## O que este caderno estabelece

- Os 23 arquivos de entrada conferem com o manifesto SHA-256 publicado.
- As 16 medianas de $F_S$ do artigo são rederiváveis dos componentes STL brutos.
- A completude (658 de 736) e as datas por classe conferem.
- A faixa de rotatividade espacial (222 a 774 km) confere.
- A Figura 1 é regenerada pelo mesmo código que produziu o artigo.

## O que este caderno não estabelece

- **Estabilidade do PPI entre sementes.** O trabalho usa a semente 13. O protocolo de consenso
  multissemente existe no repositório, mas não foi executado para esta submissão.
- **Validação independente dos rótulos TerraClass**, declarada como limitação no artigo.
- **A malha amostral a partir das máscaras de permanência**, que dependem de rasters não
- **A malha amostral a partir das máscaras de permanência do TerraClass**, que não
  acompanham este repositório.
- As etapas anteriores da cadeia — amostragem, WTSS, PPI, painéis, revisão e STL — cobertas
  pelos cadernos `standalone/01` a `standalone/06`.